In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches
from src.elo import prepare_matches
from src.simulation import simulate_match, simulate_group, simulate_group_stage, simulate_knockout_match, construct_round32, simulate_knockout_round, construct_round16, construct_QF, construct_SF, simulate_tournament
from src.features import build_features, assign_third_place_slots

#Load everything from earlier notebooks
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
features = joblib.load('model_features.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")
df_confederations = pd.read_csv('../data/FIFA_confederations.csv')

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

In [58]:
team_to_group = {team: group for group, teams in wc_groups.items() for team in teams}
team_to_confederation = dict(zip(df_confederations["nation"], df_confederations["confederation"]))

# Create group_stages equivalent
rows = []
for group, teams in wc_groups.items():
    for i, team in enumerate(teams, 1):
        rows.append({"group": group, "position": i, "nation": team})
df_groups = pd.DataFrame(rows)

df_group_fixtures = group_stage_matches.copy()
df_group_fixtures["group"] = df_group_fixtures["home_team"].map(team_to_group)

#Example run
x=build_features('Argentina', 'Brazil', country_elo, team_to_confederation, features)
x

,const,home_elo,away_elo,neutral,tournament_weight,home_confederation_AFC,home_confederation_CAF,home_confederation_CONCACAF,home_confederation_CONMEBOL,home_confederation_OFC,home_confederation_UEFA,home_confederation_Unknown,away_confederation_AFC,away_confederation_CAF,away_confederation_CONCACAF,away_confederation_CONMEBOL,away_confederation_OFC,away_confederation_UEFA,away_confederation_Unknown
0,1.0,2114.688178,1992.738734,1.0,5.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [59]:
y = simulate_match('Argentina', 'Brazil', model_h, model_a, country_elo, team_to_confederation, features)
y

{'home_team': 'Argentina',
 'away_team': 'Brazil',
 'home_goals': 1,
 'away_goals': 1,
 'result': 'draw',
 'winner': None}

In [60]:
group_a_table, group_a_matches = simulate_group("C", df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)

display(group_a_matches)
display(group_a_table)

,home_team,away_team,home_goals,away_goals,result,winner
0,Brazil,Morocco,2,3,away_win,Morocco
1,Haiti,Scotland,0,1,away_win,Scotland
2,Scotland,Morocco,1,0,home_win,Scotland
3,Brazil,Haiti,4,0,home_win,Brazil
4,Scotland,Brazil,3,3,draw,NaN
5,Morocco,Haiti,0,0,draw,NaN


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Scotland,3,2,1,0,5,3,2,7,1,C
1,Brazil,3,1,1,1,9,6,3,4,2,C
2,Morocco,3,1,1,1,3,3,0,4,3,C
3,Haiti,3,0,1,2,0,5,-5,1,4,C


In [61]:
group_stage = simulate_group_stage(df_groups, df_group_fixtures, model_h, model_a, country_elo, team_to_confederation, features)
group_stage_result = group_stage[0]

top2 = group_stage_result.groupby('group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('group').nth(2).copy() #All teams that placed third (only 8 of them move on)
best8_third = third.sort_values(
    ['points', 'goal_difference', 'goals_for'], 
    ascending=[False, False, False]
).head(8).reset_index(drop=True)

best8_third['third_place_rank'] = best8_third.index + 1

display(group_stage_result)

,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,Mexico,3,2,1,0,6,0,6,7,1,A
1,South Africa,3,2,1,0,3,1,2,7,2,A
2,South Korea,3,0,1,2,2,4,-2,1,3,A
3,Czech Republic,3,0,1,2,1,7,-6,1,4,A
4,Canada,3,2,1,0,5,1,4,7,1,B
5,Bosnia and Herzegovina,3,2,0,1,5,4,1,6,2,B
6,Switzerland,3,1,1,1,1,2,-1,4,3,B
7,Qatar,3,0,0,3,1,5,-4,0,4,B
8,Morocco,3,2,1,0,4,0,4,7,1,C
9,Brazil,3,2,0,1,8,4,4,6,2,C


In [62]:
round_of_32 = pd.concat([top2, best8_third])
round_of_32

,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group,third_place_rank
0,Mexico,3,2,1,0,6,0,6,7,1,A,NaN
1,South Africa,3,2,1,0,3,1,2,7,2,A,NaN
4,Canada,3,2,1,0,5,1,4,7,1,B,NaN
5,Bosnia and Herzegovina,3,2,0,1,5,4,1,6,2,B,NaN
8,Morocco,3,2,1,0,4,0,4,7,1,C,NaN
9,Brazil,3,2,0,1,8,4,4,6,2,C,NaN
12,United States,3,2,0,1,6,3,3,6,1,D,NaN
13,Paraguay,3,2,0,1,4,2,2,6,2,D,NaN
16,Ecuador,3,3,0,0,7,2,5,9,1,E,NaN
17,Ivory Coast,3,2,0,1,2,2,0,6,2,E,NaN


In [63]:
rd32_teams = {}
for team in top2.itertuples(index=False):
        rd32_teams[f'{team.group_rank}{team.group}'] = team.team
        
thirds = assign_third_place_slots(best8_third)
rd32_teams |= thirds

x = construct_round32(rd32_teams)
simulate_knockout_match(x[73][0], x[73][1], model_h, model_a, country_elo, team_to_confederation, features)

{'home_team': 'South Africa',
 'away_team': 'Bosnia and Herzegovina',
 'home_goals': 1,
 'away_goals': 1,
 'result': 'away_win',
 'result_type': 'OT/Pens',
 'winner': 'Bosnia and Herzegovina',
 'loser': 'South Africa',
 'home_win_prob': np.float64(0.35488152591543976),
 'draw_prob': np.float64(0.2919221740445139),
 'away_win_prob': np.float64(0.35319157885228586)}

In [64]:
winners, results = simulate_knockout_round(x, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,South Africa,Bosnia and Herzegovina,0,0,OT/Pens,South Africa,Bosnia and Herzegovina,0.354882,0.291922,0.353192
1,Ecuador,Turkey,0,1,regular_time,Turkey,Ecuador,0.438425,0.289739,0.271829
2,Japan,Brazil,0,3,regular_time,Brazil,Japan,0.291794,0.288932,0.419268
3,Morocco,Sweden,2,0,regular_time,Morocco,Sweden,0.583133,0.255459,0.161374
4,Norway,Scotland,1,0,regular_time,Norway,Scotland,0.451663,0.278406,0.269919
5,Ivory Coast,France,0,4,regular_time,France,Ivory Coast,0.106707,0.210108,0.683049
6,Mexico,Saudi Arabia,1,1,OT/Pens,Mexico,Saudi Arabia,0.731520,0.182274,0.085902
7,England,Uzbekistan,0,1,regular_time,Uzbekistan,England,0.559393,0.261132,0.179446
8,United States,Switzerland,0,1,regular_time,Switzerland,United States,0.301064,0.281505,0.417423
9,Iran,Jordan,1,0,regular_time,Iran,Jordan,0.463954,0.286301,0.249736


In [65]:
r16_matchups = construct_round16(winners)

winners, results = simulate_knockout_round(r16_matchups, model_h, model_a, country_elo, team_to_confederation, features)

display(winners)

{89: 'Turkey',
 90: 'Brazil',
 91: 'Morocco',
 92: 'Mexico',
 93: 'Austria',
 94: 'Switzerland',
 95: 'Uruguay',
 96: 'Germany'}

In [66]:
QF_matchsups = construct_QF(winners)

winners, results = simulate_knockout_round(QF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Turkey,Brazil,3,1,regular_time,Turkey,Brazil,0.194990,0.253917,0.551054
1,Austria,Switzerland,1,2,regular_time,Switzerland,Austria,0.251401,0.272501,0.476082
2,Morocco,Mexico,1,3,regular_time,Mexico,Morocco,0.342116,0.294835,0.363045
3,Uruguay,Germany,1,2,regular_time,Germany,Uruguay,0.365605,0.295666,0.338724


In [67]:
SF_matchsups = construct_SF(winners)

winners, results = simulate_knockout_round(SF_matchsups, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Turkey,Switzerland,1,2,regular_time,Switzerland,Turkey,0.326225,0.287057,0.386711
1,Mexico,Germany,0,0,OT/Pens,Germany,Mexico,0.407760,0.287620,0.304613


In [68]:
winner, results = simulate_knockout_round({103: (winners[101], winners[102])}, model_h, model_a, country_elo, team_to_confederation, features)

display(results)

print(f'{winner[103]} won the World Cup')

,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob
0,Switzerland,Germany,1,3,regular_time,Germany,Switzerland,0.264583,0.278276,0.457129


Germany won the World Cup


In [69]:
r = simulate_tournament(model_h, model_a, country_elo, team_to_confederation, features, df_groups, df_group_fixtures)
summary = r['summary']
print(f"Winner: {summary['winner']}")
print(f"loser: {summary['runner_up']}")
print(f"sf_teams: {summary['sf_teams']}")
print(f"qf_teams: {summary['qf_teams']}")

display(r['group_tables'])
display(r['group_matches'])
display(r['bracket'])


Winner: Spain
loser: Colombia
sf_teams: ['Ecuador', 'Spain', 'England', 'Colombia']
qf_teams: ['Ecuador', 'Netherlands', 'Brazil', 'England', 'Spain', 'Iraq', 'Paraguay', 'Colombia']


,team,played,wins,draws,losses,goals_for,goals_against,goal_difference,points,group_rank,group
0,South Korea,3,3,0,0,6,2,4,9,1,A
1,Mexico,3,2,0,1,8,5,3,6,2,A
2,Czech Republic,3,1,0,2,4,7,-3,3,3,A
3,South Africa,3,0,0,3,2,6,-4,0,4,A
4,Canada,3,3,0,0,7,2,5,9,1,B
5,Switzerland,3,1,1,1,2,4,-2,4,2,B
6,Bosnia and Herzegovina,3,0,2,1,0,1,-1,2,3,B
7,Qatar,3,0,1,2,3,5,-2,1,4,B
8,Brazil,3,2,0,1,4,3,1,6,1,C
9,Haiti,3,1,1,1,3,2,1,4,2,C


,home_team,away_team,home_goals,away_goals,result,winner
0,Mexico,South Africa,3,1,home_win,Mexico
1,South Korea,Czech Republic,2,1,home_win,South Korea
2,Czech Republic,South Africa,2,1,home_win,Czech Republic
3,Mexico,South Korea,1,3,away_win,South Korea
4,Mexico,Czech Republic,4,1,home_win,Mexico
...,...,...,...,...,...,...
67,Ghana,Panama,1,0,home_win,Ghana
68,England,Ghana,1,0,home_win,England
69,Panama,Croatia,1,1,draw,NaN
70,Panama,England,0,2,away_win,England


,home_team,away_team,home_goals,away_goals,result_type,winner,loser,home_win_prob,draw_prob,away_win_prob,round
0,Mexico,Switzerland,2,2,OT/Pens,Mexico,Switzerland,0.456492,0.280172,0.263325,R32
1,Ecuador,Scotland,1,0,regular_time,Ecuador,Scotland,0.586048,0.248095,0.165810,R32
2,Netherlands,Haiti,2,1,regular_time,Netherlands,Haiti,0.694177,0.190419,0.115074,R32
3,Brazil,Japan,0,0,OT/Pens,Brazil,Japan,0.425442,0.303749,0.270805,R32
4,Senegal,Sweden,1,1,OT/Pens,Senegal,Sweden,0.502964,0.280050,0.216972,R32
5,Germany,Norway,1,0,regular_time,Germany,Norway,0.321398,0.287776,0.390820,R32
6,South Korea,Saudi Arabia,3,1,regular_time,South Korea,Saudi Arabia,0.570202,0.254820,0.174942,R32
7,England,Algeria,3,1,regular_time,England,Algeria,0.574582,0.258939,0.166449,R32
8,United States,Iraq,0,3,regular_time,Iraq,United States,0.533803,0.263598,0.202573,R32
9,New Zealand,Czech Republic,2,0,regular_time,New Zealand,Czech Republic,0.832966,0.101795,0.058520,R32


In [70]:
joblib.dump(df_groups, "df_groups.joblib")
joblib.dump(df_group_fixtures, "df_groups_fixtures.joblib")
joblib.dump(team_to_confederation, "team_to_confed.joblib")

['team_to_confed.joblib']